In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

file_path = project_root / "data" / "raw" / "Dataset Capstone.xlsx"

df_branches = pd.read_excel(
    file_path,
    sheet_name="branches"
)

print("Jumlah baris dan kolom:", df_branches.shape)
display(df_branches.head())

Jumlah baris dan kolom: (3, 16)


,branch_id,branch_name,city,address,opening_time,closing_time,phone,facilities,google_maps_url,instagram_url,gofood_url,grabfood_url,is_active,source_id,verification_status,data_type
0,BR001,KOPILOGI,GARUT,"Jl. Cikuray No.42, Regol, Kec. Garut Kota, Kab...",08:00:00,22:00:00,62 813 6786 6999,"wifi, ac, indoor, outdoor, toilet, parkir",https://maps.app.goo.gl/R479crDeXQ2gdv466,https://www.instagram.com/kopilogi?dlrf=MWVpcD...,https://gofood.link/a/zwzBrsq,NaN,True,SRC001,pending_pic,actual
1,BR002,WAROENG KOPILOGI,GARUT,"Jl. Guntur No.14, Paminggir, Kec. Garut Kota, ...",07:00:00,22:00:00,62 813 1222 0434,"wifi, ruang vip, karaoke, toilet, smoking area...",https://maps.app.goo.gl/DU4ZTnNwDtow4rj48,https://www.instagram.com/waroengkopilogi_garu...,https://gofood.link/a/BbAS6m1,NaN,True,SRC002,pending_pic,actual
2,BR003,KOPILOGI TASIK,TASIKMALAYA,"Jl. Dr. Sukarjo, Tawangsari, Kec. Tawang, Kota...",08:00:00,23:00:00,62 812 2228 1554,"wifi, ac, smoking area, mushola, toilet, slow ...",https://maps.app.goo.gl/tB6dk6wVaDYa7vzf8,https://www.instagram.com/kopilogi.tasik?cplk=...,https://gofood.link/a/LU2VaiJ,NaN,True,SRC003,pending_pic,actual


In [2]:
print("=== STRUKTUR DATASET CABANG ===")
df_branches.info()

=== STRUKTUR DATASET CABANG ===
<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   branch_id            3 non-null      str    
 1   branch_name          3 non-null      str    
 2   city                 3 non-null      str    
 3   address              3 non-null      str    
 4   opening_time         3 non-null      object 
 5   closing_time         3 non-null      object 
 6   phone                3 non-null      str    
 7   facilities           3 non-null      str    
 8   google_maps_url      3 non-null      str    
 9   instagram_url        3 non-null      str    
 10  gofood_url           3 non-null      str    
 11  grabfood_url         0 non-null      float64
 12  is_active            3 non-null      bool   
 13  source_id            3 non-null      str    
 14  verification_status  3 non-null      str    
 15  data_type            3 

In [3]:
missing_report = pd.DataFrame({
    "jumlah_kosong": df_branches.isna().sum(),
    "persentase_kosong": (
        df_branches.isna().mean() * 100
    ).round(2)
})

display(missing_report)

,jumlah_kosong,persentase_kosong
branch_id,0,0.0
branch_name,0,0.0
city,0,0.0
address,0,0.0
opening_time,0,0.0
closing_time,0,0.0
phone,0,0.0
facilities,0,0.0
google_maps_url,0,0.0
instagram_url,0,0.0


In [4]:
print("=== PEMERIKSAAN DUPLIKASI ===")

print(
    "Duplikasi seluruh baris:",
    df_branches.duplicated().sum()
)

print(
    "Duplikasi branch_id:",
    df_branches["branch_id"].duplicated().sum()
)

print(
    "Duplikasi nama cabang:",
    df_branches["branch_name"]
    .astype("string")
    .str.strip()
    .str.lower()
    .duplicated()
    .sum()
)

=== PEMERIKSAAN DUPLIKASI ===
Duplikasi seluruh baris: 0
Duplikasi branch_id: 0
Duplikasi nama cabang: 0


In [5]:
print("=== STATUS VERIFIKASI CABANG ===")

display(
    df_branches["verification_status"]
    .value_counts(dropna=False)
)

print("\n=== STATUS CABANG AKTIF ===")

display(
    df_branches["is_active"]
    .value_counts(dropna=False)
)

=== STATUS VERIFIKASI CABANG ===


verification_status
pending_pic    3
Name: count, dtype: int64


=== STATUS CABANG AKTIF ===


is_active
True    3
Name: count, dtype: int64

In [6]:
print("=== PEMERIKSAAN JAM OPERASIONAL ===")

display(
    df_branches[
        ["branch_id", "branch_name",
         "opening_time", "closing_time"]
    ]
)

print("\nTipe nilai jam buka:")
print(df_branches["opening_time"].map(type).value_counts())

print("\nTipe nilai jam tutup:")
print(df_branches["closing_time"].map(type).value_counts())

=== PEMERIKSAAN JAM OPERASIONAL ===


,branch_id,branch_name,opening_time,closing_time
0,BR001,KOPILOGI,08:00:00,22:00:00
1,BR002,WAROENG KOPILOGI,07:00:00,22:00:00
2,BR003,KOPILOGI TASIK,08:00:00,23:00:00



Tipe nilai jam buka:
opening_time
<class 'datetime.time'>    3
Name: count, dtype: int64

Tipe nilai jam tutup:
closing_time
<class 'datetime.time'>    3
Name: count, dtype: int64


In [7]:
from urllib.parse import urlparse
import pandas as pd

print("=== PEMERIKSAAN FORMAT URL ===")

url_columns = [
    "google_maps_url",
    "instagram_url",
    "gofood_url",
    "grabfood_url"
]

def check_url(url):
    if pd.isna(url) or str(url).strip() == "":
        return "missing"

    parsed = urlparse(str(url).strip())

    if parsed.scheme in ["http", "https"] and parsed.netloc:
        return "valid_format"

    return "invalid_format"

url_report = pd.DataFrame({
    col: df_branches[col].apply(check_url).value_counts()
    for col in url_columns
}).fillna(0).astype(int)

display(url_report)

=== PEMERIKSAAN FORMAT URL ===


,google_maps_url,instagram_url,gofood_url,grabfood_url
missing,0,0,0,3
valid_format,3,3,3,0


In [8]:
print("=== PEMERIKSAAN KONSISTENSI CABANG ===")

columns = [
    "branch_id",
    "branch_name",
    "city",
    "is_active",
    "source_id",
    "verification_status",
    "data_type"
]

display(df_branches[columns])

print("\nDaftar kota:")
display(df_branches["city"].value_counts(dropna=False))

print("\nJenis data:")
display(df_branches["data_type"].value_counts(dropna=False))

=== PEMERIKSAAN KONSISTENSI CABANG ===


,branch_id,branch_name,city,is_active,source_id,verification_status,data_type
0,BR001,KOPILOGI,GARUT,True,SRC001,pending_pic,actual
1,BR002,WAROENG KOPILOGI,GARUT,True,SRC002,pending_pic,actual
2,BR003,KOPILOGI TASIK,TASIKMALAYA,True,SRC003,pending_pic,actual



Daftar kota:


city
GARUT          2
TASIKMALAYA    1
Name: count, dtype: int64


Jenis data:


data_type
actual    3
Name: count, dtype: int64

In [9]:
from pathlib import Path
import pandas as pd

reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

branch_issues = pd.DataFrame([
    {
        "issue_id": "BQ001",
        "column": "grabfood_url",
        "issue": "Tiga nilai kosong",
        "affected_records": 3,
        "action": "Pertahankan kosong sampai tersedia sumber",
        "status": "pending"
    },
    {
        "issue_id": "BQ002",
        "column": "verification_status",
        "issue": "Seluruh cabang menunggu verifikasi PIC",
        "affected_records": 3,
        "action": "Konfirmasi informasi kepada pihak Kopilogi",
        "status": "pending"
    },
    {
        "issue_id": "BQ003",
        "column": "opening_time, closing_time",
        "issue": "Format waktu perlu disiapkan untuk CSV/JSON",
        "affected_records": 3,
        "action": "Standardisasi menjadi HH:mm",
        "status": "planned"
    },
    {
        "issue_id": "BQ004",
        "column": "branch_name, city",
        "issue": "Penulisan menggunakan huruf kapital",
        "affected_records": 3,
        "action": "Standardisasi format tampilan",
        "status": "planned"
    },
    {
        "issue_id": "BQ005",
        "column": "url_columns",
        "issue": "Tujuan tautan belum diperiksa",
        "affected_records": 9,
        "action": "Lakukan pemeriksaan manual setiap tautan",
        "status": "pending"
    }
])

output_path = reports_dir / "branches_quality_report.csv"

branch_issues.to_csv(output_path, index=False)

display(branch_issues)
print("Laporan disimpan:", output_path)

,issue_id,column,issue,affected_records,action,status
0,BQ001,grabfood_url,Tiga nilai kosong,3,Pertahankan kosong sampai tersedia sumber,pending
1,BQ002,verification_status,Seluruh cabang menunggu verifikasi PIC,3,Konfirmasi informasi kepada pihak Kopilogi,pending
2,BQ003,"opening_time, closing_time",Format waktu perlu disiapkan untuk CSV/JSON,3,Standardisasi menjadi HH:mm,planned
3,BQ004,"branch_name, city",Penulisan menggunakan huruf kapital,3,Standardisasi format tampilan,planned
4,BQ005,url_columns,Tujuan tautan belum diperiksa,9,Lakukan pemeriksaan manual setiap tautan,pending


Laporan disimpan: a:\ASAH DICODING\CAPSTONE PROJECT\kopilogi-data-science\reports\branches_quality_report.csv
